In [1]:
import duckdb

import os

from torchgen.api.cpp import return_type
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64

import xgboost as xgb
import pandas as pd

import pyarrow as pa
from duckdb.typing import *

In [2]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch.nn.Sequential(
            #Input = 3 x 32 x 32, Output = 32 x 32 x 32
            #new input 3x600x375 output=
            torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 32 x 32 x 32, Output = 32 x 16 x 16
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 32 x 16 x 16, Output = 64 x 16 x 16
            torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 16 x 16, Output = 64 x 8 x 8
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            #Input = 64 x 8 x 8, Output = 64 x 8 x 8
            torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
            torch.nn.ReLU(),
            #Input = 64 x 8 x 8, Output = 64 x 4 x 4
            torch.nn.MaxPool2d(kernel_size=2),

            torch.nn.Flatten(),

            torch.nn.Linear(96*11*18, 128),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(128),




            #torch.nn.Linear(128, 128),
            #torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(128),



            torch.nn.Linear(128, 2),
            torch.nn.ReLU(),
            #torch.nn.BatchNorm1d(2),
            #torch.nn.Dropout(0.2),


            torch.nn.Softmax(dim=1)

        )

    def forward(self, x):



        return self.model(x)

In [4]:
def predict_tensor(base64image):

    class CNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = torch.nn.Sequential(
                #Input = 3 x 32 x 32, Output = 32 x 32 x 32
                #new input 3x600x375 output=
                torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 32 x 32 x 32, Output = 32 x 16 x 16
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 32 x 16 x 16, Output = 64 x 16 x 16
                torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 16 x 16, Output = 64 x 8 x 8
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                torch.nn.Flatten(),

                torch.nn.Linear(96*11*18, 128),
                torch.nn.ReLU(),
                torch.nn.BatchNorm1d(128),




                #torch.nn.Linear(128, 128),
                #torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(128),



                torch.nn.Linear(128, 2),
                torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(2),
                #torch.nn.Dropout(0.2),


                torch.nn.Softmax(dim=1)

            )

        def forward(self, x):



            return self.model(x)

    img_data = base64.b64decode(base64image)
    image = Image.open(BytesIO(img_data))
    tensor = ToTensor()(image).float()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CNN().to(device)
    model.load_state_dict(torch.load('/models/mymodel', map_location=torch.device(device)))
    model.eval() # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)

    return float(y_pred)






In [2]:
con = duckdb.connect("../duckdatabase/idnet.db")

In [43]:
#con.remove_function('predict_with_torch_naive')

In [5]:
con.create_function(
    name='predict_with_torch_naive',
    function=predict_tensor,
    parameters=['STRING'],
    return_type='FLOAT'
)

In [6]:
con.sql('select predict_with_torch_naive(imagedata) from idnet')

┌─────────────────────────────────────┐
│ predict_with_torch_naive(imagedata) │
│                float                │
├─────────────────────────────────────┤
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                 0.0 │
│                                  ·  │
│                                  ·  │
│                                  ·  │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │
│                                 1.0 │


In [2]:
xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
xgboostmodel.load_model('/models/xgboost_model.model')

In [7]:
def predict_xgboost(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs):

    xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
    xgboostmodel.load_model('/models/xgboost_model.model')

    data = [
        {
            'gender': gender,
            'ethnicity': ethnicity,
            'class': driver_class,
            'eye_color': eye_color,
            'hair_color': hair_color,
            'is_donor': is_donor,
            'is_veteran': is_veteran,
            'height_inches': height_inches,
            'weight_lbs': weight_lbs
        }
    ]
    df = pd.DataFrame(data)

    df['gender'] = df['gender'].astype('category')
    df['ethnicity'] = df['ethnicity'].astype('category')
    df['class'] = df['class'].astype('category')
    df['eye_color'] = df['eye_color'].astype('category')
    df['hair_color'] = df['hair_color'].astype('category')
    df['is_donor'] = df['is_donor'].astype('category')
    df['is_veteran'] = df['is_veteran'].astype('category')
    df['height_inches'] = df['height_inches'].astype('int64')
    df['weight_lbs'] = df['weight_lbs'].astype('int64')

    #feature_np = np.array(feature_list, dtype=object)

    y_pred = xgboostmodel.predict(df)
    #print(y_pred[0])

    return float(y_pred[0])

In [49]:
#con.remove_function('predict_with_xgboost_naive')

In [8]:
con.create_function(
    name='predict_with_xgboost_naive',
    function=predict_xgboost,
    parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER', 'VARCHAR'],
    return_type='FLOAT'
)

In [9]:
con.sql('select predict_with_xgboost_naive(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet')

┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ predict_with_xgboost_naive(gender, ethnicity, "class", eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) │
│                                                             float                                                              │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                                            0.0 │
│                                                                                                                            0.0 │
│                                                                                                                            0.0 │
│                                                                                  

In [10]:
df = con.sql('select predict_with_xgboost_naive(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet').df()
df

,"predict_with_xgboost_naive(gender, ethnicity, ""class"", eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs)"
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
993,0.0
994,0.0
995,0.0
996,0.0


In [11]:
def v_predict_tensor(base64image):

    class CNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = torch.nn.Sequential(
                #Input = 3 x 32 x 32, Output = 32 x 32 x 32
                #new input 3x600x375 output=
                torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 32 x 32 x 32, Output = 32 x 16 x 16
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 32 x 16 x 16, Output = 64 x 16 x 16
                torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 16 x 16, Output = 64 x 8 x 8
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                torch.nn.Flatten(),

                torch.nn.Linear(96*11*18, 128),
                torch.nn.ReLU(),
                torch.nn.BatchNorm1d(128),




                #torch.nn.Linear(128, 128),
                #torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(128),



                torch.nn.Linear(128, 2),
                torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(2),
                #torch.nn.Dropout(0.2),


                torch.nn.Softmax(dim=1)

            )

        def forward(self, x):
            return self.model(x)

    #base64image_decode = base64image.apply(lambda x: base64.b64decode(x))
    #image_pil = base64image_decode.apply(lambda x: Image.open(BytesIO(x)))
    #tensor_series = image_pil.apply(lambda x: ToTensor()(x).float())

    input_data = pd.Series(base64image)

    tensor_series = input_data.apply(lambda x: ToTensor()(Image.open(BytesIO(base64.b64decode(x)))).float())
    input_tensor = tensor_series.tolist()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CNN().to(device)
    model.load_state_dict(torch.load('/models/mymodel', map_location=torch.device(device)))
    model.eval() # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation for inference
        predictions = model(torch.stack(input_tensor).to(device))
        _, y_pred = torch.max(predictions.data, 1)
    #return y_pred
    return pa.array(y_pred.squeeze().tolist())

In [12]:
con.create_function(
    name='predict_with_torch_optimized',
    function=v_predict_tensor,
    parameters=['STRING'],
    return_type='FLOAT',
    type='arrow'
)

In [13]:
con.sql('select predict_with_torch_optimized(imagedata) from idnet')

┌─────────────────────────────────────────┐
│ predict_with_torch_optimized(imagedata) │
│                  float                  │
├─────────────────────────────────────────┤
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                      ·  │
│                                      ·  │
│                                      ·  │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│                               

In [14]:
def predict_xgboost_vector(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches,
                           weight_lbs):


    xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
    xgboostmodel.load_model('/models/xgboost_model.model')


    gender_ps =  pd.Series(gender, dtype="category")
    ethnicity_ps = pd.Series(ethnicity, dtype="category")
    class_ps =  pd.Series(driver_class, dtype="category")
    eye_color_ps =  pd.Series(eye_color, dtype="category")
    hair_color_ps =  pd.Series(hair_color, dtype="category")
    is_donor_ps =  pd.Series(is_donor, dtype="category")
    is_veteran_ps =  pd.Series(is_veteran, dtype="category")
    height_inches_ps =  pd.Series(height_inches, dtype="int64")
    weight_lbs_ps =  pd.Series(weight_lbs).astype("int64")


    df = pd.DataFrame( {
        'gender': gender_ps,
        'ethnicity': ethnicity_ps,
        'class': class_ps,
        'eye_color': eye_color_ps,
        'hair_color': hair_color_ps,
        'is_donor': is_donor_ps,
        'is_veteran': is_veteran_ps,
        'height_inches': height_inches_ps,
        'weight_lbs': weight_lbs_ps
        })






    y_pred = xgboostmodel.predict(df)


    return pa.array(y_pred)

In [101]:
con.remove_function('predict_with_xgboost_optimized')

In [15]:
con.create_function(
    name='predict_with_xgboost_optimized',
    function=predict_xgboost_vector,
    parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER', 'VARCHAR'],
    return_type='FLOAT',
    type='arrow'
)

In [16]:
con.sql('select predict_with_xgboost_optimized(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet').show()

┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ predict_with_xgboost_optimized(gender, ethnicity, "class", eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) │
│                                                               float                                                                │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                                                0.0 │
│                                                                                                                                0.0 │
│                                                                                                                                0.0 │
│                                                      

In [17]:
df = con.sql('select name , gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs, predict_with_xgboost_optimized(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as pred from idnet').df()
df[df['pred'].isnull()]

,name,gender,ethnicity,class,eye_color,hair_color,is_donor,is_veteran,height_inches,weight_lbs,pred
799,generated_fake_2_95,F,None,A,BRO,None,FALSE,FALSE,71,128,NaN
800,generated_fake_2_37,M,None,D,BRO,None,FALSE,FALSE,60,142,NaN
801,generated_fake_2_76,F,None,C,BRO,None,FALSE,FALSE,79,127,NaN
802,generated_fake_2_73,F,None,C,BRO,None,FALSE,FALSE,71,143,NaN
803,generated_fake_2_59,F,None,B,BRO,None,FALSE,FALSE,60,129,NaN
...,...,...,...,...,...,...,...,...,...,...,...
894,generated_fake_2_6,M,None,B,BRO,None,FALSE,FALSE,95,186,NaN
895,generated_fake_2_24,M,None,C,BRO,None,FALSE,FALSE,61,235,NaN
896,generated_fake_2_92,M,None,B,BRO,None,FALSE,FALSE,72,216,NaN
897,generated_fake_2_81,F,None,D,BRO,None,FALSE,FALSE,67,146,NaN


In [18]:
con.close()